# FarmLens — Step 1: Translate KCC to Hindi (NLLB)

KCC (Kisan Call Centre) Q&A is English/Hinglish. This notebook translates it to **Hindi** with Meta's **NLLB-200** (`distilled-600M`) and saves `kcc_translated.csv`.

> **Why NLLB and not IndicTrans2?** IndicTrans2 needs `IndicTransToolkit`, which breaks on Kaggle's current `transformers` (the `PreTrainedTokenizerBase` import error). NLLB runs on plain `transformers`, so there's nothing to conflict.

> **Why a separate notebook?** Translation is a slow, one-time step. Doing it here and saving the result as a dataset means Step 2 (fine-tuning) just reads the CSV — you never re-translate.

## Before you run
1. **Settings → Accelerator → GPU** (T4).
2. **Add Input** → your KCC CSV. Set `KCC_CSV_PATH` + column names in Config.
3. After it finishes: **Output panel → New Dataset** to save `kcc_translated.csv` as a Kaggle Dataset — Step 2 reads it from there.

In [ ]:
%%capture
# NLLB (Meta) translates via plain transformers — no IndicTransToolkit, so none of
# the PreTrainedTokenizerBase / transformers-version conflicts. Just speed up downloads.
!pip install -q hf_transfer

import os
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"               # faster model download
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"  # less fragmentation / OOM

In [ ]:
# ───────────────────────── Config ─────────────────────────
KCC_CSV_PATH = "/kaggle/input/kisan-call-center/kcc.csv"  # <-- CHANGE to your path
COL_QUESTION = "QueryText"   # farmer question column
COL_ANSWER   = "KccAns"      # answer column
MIN_ANSWER_LEN = 15          # drop junk / too-short answers
MAX_ROWS       = 20000       # cap for a fast first run (set None for all)

# Translation target
TGT_LANG    = "hin_Deva"   # hin_Deva | pan_Guru | mar_Deva | tel_Telu | tam_Taml
TRANS_BATCH = 8            # NLLB has a 256k vocab — beam search is memory-heavy.
NUM_BEAMS   = 5            #   batch*beams sequences run at once. Lower batch if OOM,
                           #   raise it (e.g. 16) if you have headroom. Beams=1 = ~5x faster.

OUTPUT_CSV  = "/kaggle/working/kcc_translated.csv"

## 1. Load and clean the KCC data (English)

KCC rows are noisy — drop empty / very short / placeholder answers and duplicates.

In [ ]:
import pandas as pd

df = pd.read_csv(KCC_CSV_PATH)
df = df[[COL_QUESTION, COL_ANSWER]].dropna()
df.columns = ["question", "answer"]
df["question"] = df["question"].astype(str).str.strip()
df["answer"]   = df["answer"].astype(str).str.strip()

# light cleaning of noisy KCC rows
df = df[df["answer"].str.len() >= MIN_ANSWER_LEN]
df = df[df["question"].str.len() >= 5]
junk = {"test", "wrong number", "general", "n/a", "na", "-", "nil"}
df = df[~df["answer"].str.lower().isin(junk)]
df = df.drop_duplicates(subset=["question", "answer"])
if MAX_ROWS:
    df = df.sample(min(MAX_ROWS, len(df)), random_state=3407).reset_index(drop=True)

print(f"Rows after cleaning: {len(df)}")
df.head(3)

## 2. Translate English → Hindi with NLLB

Translates both questions and answers via `facebook/nllb-200-distilled-600M`. The target language is set by `TGT_LANG` (NLLB uses the same `hin_Deva`-style codes).

> ⏱️ ~20k rows (40k sentences, beam=5) takes **~20–40 min on a T4**. For a quick test, set `MAX_ROWS = 2000` in Config, or `NUM_BEAMS = 1`.

In [ ]:
import torch
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
NLLB = "facebook/nllb-200-distilled-600M"
mt_tok = AutoTokenizer.from_pretrained(NLLB, src_lang="eng_Latn")
mt_model = AutoModelForSeq2SeqLM.from_pretrained(
    NLLB, torch_dtype=torch.float16
).to(DEVICE)
# NLLB forces the output language via this BOS token
TGT_BOS = mt_tok.convert_tokens_to_ids(TGT_LANG)


def translate(sentences, bs=TRANS_BATCH):
    out = []
    for i in range(0, len(sentences), bs):
        chunk = [s if s.strip() else "." for s in sentences[i:i + bs]]
        enc = mt_tok(
            chunk, truncation=True, padding=True,
            return_tensors="pt", max_length=256,
        ).to(DEVICE)
        with torch.no_grad():
            gen = mt_model.generate(
                **enc, forced_bos_token_id=TGT_BOS,
                num_beams=NUM_BEAMS, max_length=256,
            )
        out.extend(mt_tok.batch_decode(gen, skip_special_tokens=True))
        print(f"  {min(i + bs, len(sentences))}/{len(sentences)}", end="\r")
    return out


print("Translating questions…")
df["question"] = translate(df["question"].tolist())
print("\nTranslating answers…")
df["answer"] = translate(df["answer"].tolist())

df.to_csv(OUTPUT_CSV, index=False)
print("\nSaved translated data to", OUTPUT_CSV)
df.head(3)

## 3. Save as a Kaggle Dataset

`kcc_translated.csv` is now in this notebook's **Output**. To use it in Step 2:

1. **Output panel** (right side) → find `kcc_translated.csv`.
2. Click **New Dataset** → name it e.g. `kcc-hindi-translated` → **Create**.
3. In **Step 2** (the fine-tune notebook), **Add Input** → search that dataset name.

You only translate once — then iterate on training as much as you like.